## Annual Technology Baseline Reporting

### Offshore Wind

In [1]:
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format

from time import perf_counter
from pathlib import Path

from waves import Project
from waves.utilities import load_yaml
from waves.utilities import check_ref_sites

from waves.utilities.atb_tools import breakdown_plots

import matplotlib.pyplot as plt


### Check config files based on reference site table.

In [2]:
year = 2023
library_path = Path(f"../library/atb_base_{year}/")

config_path = library_path / "project/config"
ref_site_df = check_ref_sites(config_path, "reference sites.xlsx", verbose=False)

ref_site_df

TypeError: list indices must be integers or slices, not str

#### Initialize the projects from dictionary configuration files. 

In [ ]:
site_names = []
project_list = []

for _, row in ref_site_df.iterrows():

    base = f"Site{row['Site']}_atb_{year}.yaml"
    site_names.append(base.split('_')[0])

    config = load_yaml(config_path, base)

    config.update({"library_path" : library_path,
                  # "orbit_config" : Path(atb_path / "orbit_config" / f"Site{row['Site']}_atb_{year}_install.yaml").resolve(),
                  # "wombat_config" : Path(atb_path / "wombat_config" / f"Site{row['Site']}_atb_{year}_operations.yaml").resolve(),
                  # "floris_config" : Path(atb_path / "floris_config" / f"Site{row['Site']}_atb_{year}_floris_jensen.yaml").resolve(),
                    }
    )

    start = perf_counter()

    project = Project.from_dict(config)
    project_list.append(project)

    end = perf_counter()

    print(f"{base} loading time: {(end-start):.2f} seconds")


#### Run Project

In [ ]:
XX
id = len(project_list)
for i,p in enumerate(project_list[:id]):

    start1 = perf_counter()
    p.run(
        full_wind_rose=False,  # use the WOMBAT date range
    )
    p.wombat.env.cleanup_log_files()  # Delete logging data from the WOMBAT simulations

    end1 = perf_counter()

    print(f"{site_names[i]} run time: {end1 - start1:,.2f} seconds")

### Simple Report Generation

In [ ]:
report_df_final = pd.DataFrame()
metrics_configuration = {
    "Project Capacity (MW)": {
        "metric": "capacity",
        "kwargs": {"units": "mw"}
    },
    "CapEx (M$)": {
        "metric": "capex",
        "kwargs": {"million": True,}
    },
    "CapEx per kW ($/kW)": {
        "metric": "capex",
        "kwargs": {"per_capacity": "kw"}
    },
    "Soft CapEx (M$)" :{
        "metric": "soft_capex",
        "kwargs": {"million": True,}
    },
    "Soft CapEx per kW ($/kW)": {
        "metric": "soft_capex",
        "kwargs": {"per_capacity": "kw",}
    },
    "OpEx per kW ($/kW)": {
        "metric": "opex",
        "kwargs": {"per_capacity": "kw",
                   }},
    "AEP (MWh)": {
        "metric": "energy_production",
        "kwargs": {"units": "mw", "aep": True }
    },
    "Net Capacity Factor With All Losses (%)": {
        "metric": "capacity_factor",
        "kwargs": {"which": "net", }
    },
    "LCOE ($/MWh)": {"metric": "lcoe"},
}

for i,p in enumerate(project_list[:id]):

# Generate the reports using WAVES and the above configurations
# NOTE: the results are transposed to view them more easily for the example, otherwise
# each row would be a project, which is helpful for combining the results of many scenarios

    report_df = p.generate_report(metrics_configuration, site_names[i])

    n_years = p.operations_years

    report_df["Annual OpEx per kW ($/kW-year)"] = report_df["OpEx per kW ($/kW)"] / n_years

    report_df.pop('OpEx per kW ($/kW)')
# Combine all reports into one, easy to view dataframe
    report_df_final = report_df_final.join(report_df.T, how="outer",).fillna(0.0)

    report_df_final.index.name = "Metrics"

report_df_final.to_csv("reference_sites_report.csv")

report_df_final

In [ ]:
row = "LCOE ($/MWh)"
report_df_final.loc[row].plot(kind='bar', rot=45,label='ATB Sites (preliminary)')
plt.axhline(100, color='black', linestyle='--', linewidth=2, label='NRWAL-Fixed Bottom (placeholder)')
plt.axhline(145, color='black', linestyle=':', linewidth=2, label='NRWAL-Floating (placeholder)')

plt.ylabel(row)
plt.legend(bbox_to_anchor=(1.65, 0.3), loc='lower right', reverse=True)

In [ ]:
def stacked_bar_chart(df, transpose=False, lpos=(1.5,0.5)):

    if transpose:
        df = df.T
        xticks = df.index
        ylabel = df.columns.name
    else:
        df = df
        xticks = df.columns
        ylabel = df.index.name

    df.plot(kind='bar', stacked=True)
    plt.xticks(rotation=45)
    plt.ylabel(ylabel)
    #plt.xlabel("ATB Site")

    plt.legend(bbox_to_anchor=lpos, loc='lower right', reverse=True)
    #plt.title(df.index.name)

    #plt.legend()

In [ ]:
# Capture the CapEx breakdown for each reference site

report_capex_df = pd.DataFrame()

for i,p in enumerate(project_list[:id]):

    _temp_df = p.capex(breakdown=True,per_capacity="kw").drop(columns=["CapEx"])
    _temp_df = p.capex(breakdown=True)
    #print(_temp_df)
    _temp_df.columns = [f"{site_names[i]}"]

    #_temp_df[f"{site_names[i]} CapEx ($kW)"] = _temp_df[f"{site_names[i]} CapEx ($)"] / p.capacity("kw")

    report_capex_df = report_capex_df.join(_temp_df, how="outer").fillna(0.0)

report_capex_df = report_capex_df / 1e6
report_capex_df.index.name = "CapEx (M$)"

report_capex_df.to_csv("reference_sites_capex_report.csv")

stacked_bar_chart(report_capex_df.drop("Total"), transpose=True, lpos=(1.6, 0.1))

report_capex_df


In [ ]:
# Capture the Soft CapEx breakdown for each reference site

report_soft_capex_df = pd.DataFrame()

for i,p in enumerate(project_list[:id]):

    _temp_df = p.soft_capex(breakdown=True,per_capacity="kw").drop(columns=["Soft CapEx"])
    _temp_df = p.soft_capex(breakdown=True)
    _temp_df.columns = [f"{site_names[i]}"]

    #_temp_df[f"{site_names[i]} Soft CapEx ($kW)"] = _temp_df[f"{site_names[i]} Soft CapEx ($)"] / p.capacity("kw")

    report_soft_capex_df = report_soft_capex_df.join(_temp_df, how="outer").fillna(0.0)

report_soft_capex_df = report_soft_capex_df / 1e6
report_soft_capex_df.index.name = "CapEx (M$)"

report_soft_capex_df.to_csv("reference_sites_soft_capex_report.csv")

stacked_bar_chart(report_soft_capex_df.drop("Total"), transpose=True,lpos=(1.5, 0.3))

report_soft_capex_df


In [ ]:
# Capture the Losses breakdown for each reference site

losses_report_df = pd.DataFrame()

for i,p in enumerate(project_list[:id]):

    _temp_df = p.loss_ratio(breakdown=True)
    _temp_df.columns =[f"{site_names[i]}"]

    losses_report_df = losses_report_df.join(_temp_df, how='outer').fillna(0.0)
    losses_report_df.index.name = "Losses (%)"

stacked_bar_chart(losses_report_df, transpose=True,lpos=(1.45, 0.2))

losses_report_df.to_csv("reference_sites_losses_report.csv")

losses_report_df
